In [1]:
# --- BƯỚC 0: CÀI ĐẶT & CẤU HÌNH MÔI TRƯỜNG (CHẠY ĐẦU TIÊN) ---
!pip install -q keras_cv_attention_models

import os
# [QUAN TRỌNG] Ép buộc sử dụng Keras 2 (Legacy) để tương thích với thư viện Swin
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Kiểm tra xem đã về đúng Keras 2 chưa (Nếu output có chữ 'tf.keras' hoặc version kết thúc bằng -tf là đúng)
print(f"TensorFlow Version: {tf.__version__}")
try:
    import tf_keras
    print(f"Running with Legacy Keras: {tf_keras.__version__}")
except:
    print("Running with TensorFlow internal Keras (Legacy mode active)")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Import Swin Transformer
from keras_cv_attention_models import swin_transformer_v2
from sklearn.metrics import accuracy_score, recall_score, f1_score

# --- 1. CẤU HÌNH ---
BATCH_SIZE = 32 
SPLIT_VAL = 0.2
N_TEST_IMAGES = 3000
EPOCHS = 10
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3


2026-01-03 23:59:50.029192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767484790.050277     182 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767484790.056713     182 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767484790.073995     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767484790.074015     182 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767484790.074017     182 computation_placer.cc:177] computation placer alr

TensorFlow Version: 2.19.0
Running with Legacy Keras: 2.19.0


In [3]:
# --- 2. DATASET WRAPPER (Giữ nguyên) ---
class celeb_dataframe():
    def __init__(self, main_folder="/kaggle/input/celeba-dataset", selected_features=[], drop_features=[]):
        self.main_folder = main_folder
        self.images_folder = os.path.join(main_folder, 'img_align_celeba/img_align_celeba')
        self.attr_path = os.path.join(main_folder, "list_attr_celeba.csv")
        self.features_name = []
        self.prepare(drop_features, selected_features)
        
    def prepare(self, drop_features, selected_features):
        if len(selected_features) == 0:
            self.attributes = pd.read_csv(self.attr_path)
            self.num_features = 40
        else:
            self.num_features = len(selected_features)
            selected_features = selected_features.copy()
            selected_features.append("image_id")
            self.attributes = pd.read_csv(self.attr_path)[selected_features]
        
        if len(drop_features) != 0:
            for feature in drop_features:
                if feature in self.attributes:
                    self.attributes = self.attributes.drop(feature, axis=1)
                    self.num_features -= 1
            
        self.attributes.set_index("image_id", inplace=True)
        self.attributes.replace(to_replace=-1, value=0, inplace=True)
        self.attributes["image_id"] = list(self.attributes.index)
        self.features_name = list(self.attributes.columns)[:-1]
        return self.attributes

# --- 3. PREPARE DATA ---
features = [
    'Male', 'Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair',
    'Straight_Hair', 'Wavy_Hair', 'Receding_Hairline', 'Mustache', 'Sideburns', 
    'No_Beard', 'Eyeglasses', 'Wearing_Hat', 'Wearing_Earrings', 'Wearing_Necklace', 
    'Wearing_Necktie'
]

celeb_df = celeb_dataframe()
full_celeb_data = celeb_df.prepare(drop_features=[], selected_features=features)

# Tách dữ liệu
test_df = full_celeb_data.tail(N_TEST_IMAGES)
remaining_df = full_celeb_data.iloc[:-N_TEST_IMAGES]
split_ind = int((1 - SPLIT_VAL) * remaining_df.shape[0])
train_df = remaining_df[:split_ind]
validation_df = remaining_df[split_ind:]

print(f"Train: {train_df.shape[0]}, Val: {validation_df.shape[0]}, Test: {test_df.shape[0]}")

# --- 4. DATA GENERATORS ---
train_datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=20, 
    width_shift_range=0.2, 
    height_shift_range=0.2, 
    shear_range=0.2, 
    zoom_range=0.2, 
    horizontal_flip=True, 
    fill_mode='nearest'
)
valid_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=True
)

validation_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=validation_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

test_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=test_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

Train: 159679, Val: 39920, Test: 3000
Found 159679 validated image filenames.
Found 39920 validated image filenames.
Found 3000 validated image filenames.


In [4]:
# --- 5. BUILD MODEL (ĐÃ FIX LỖI ATTRIBUTE VÀ TENSOR) ---
def build_swin_classifier(num_features):
    # Sử dụng đúng tên hàm cho size 224x224 (window8)
    # Vì đã set TF_USE_LEGACY_KERAS=1, lỗi ValueError sẽ biến mất
    base = swin_transformer_v2.SwinTransformerV2Tiny_window8(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
        num_classes=0, 
        pretrained=None 
    )

    x = base.output
    # Xử lý output shape của Swin
    if len(x.shape) == 4: # (Batch, H, W, C)
         x = GlobalAveragePooling2D()(x)
    elif len(x.shape) == 3: # (Batch, Tokens, C)
         x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    output = Dense(num_features, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output, name="SwinFace_Attribute")
    return model

model = build_swin_classifier(len(celeb_df.features_name))


I0000 00:00:1767484916.500609     182 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1767484916.504502     182 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
# Compile model
# Transformer thường cần learning rate nhỏ hơn CNN.
# Sử dụng AdamW (Adam with Weight Decay) thường tốt hơn cho Transformer.
model.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), metrics=['binary_accuracy'])


In [6]:
model.summary() # Bỏ comment nếu muốn xem cấu trúc (rất dài)


Model: "SwinFace_Attribute"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 stem_conv (Conv2D)          (None, 56, 56, 96)           4704      ['input_1[0][0]']             
                                                                                                  
 stem_ln (LayerNormalizatio  (None, 56, 56, 96)           192       ['stem_conv[0][0]']           
 n)                                                                                               
                                                                                                  
 tf.reshape (TFOpLambda)     (None, 8, 7, 768)            0         ['stem_ln[0][

In [7]:
# --- 6. TRAIN ---
os.makedirs("/kaggle/working/ckpts_swin/", exist_ok=True)
earlystop = EarlyStopping(monitor="val_binary_accuracy", patience=4, restore_best_weights=True)
model_checkpoint = ModelCheckpoint(
    filepath="/kaggle/working/ckpts_swin/weights-SwinV2-{val_binary_accuracy:.4f}.h5", # Đuôi .h5 chuẩn cho legacy
    save_weights_only=True, monitor='val_binary_accuracy', mode='max', save_best_only=True, verbose=1
)

history = model.fit(
    train_generator, epochs=EPOCHS, steps_per_epoch=len(train_generator),
    validation_data=validation_generator, validation_steps=len(validation_generator),
    callbacks=[earlystop, model_checkpoint], verbose=1
)

# --- 7. EVALUATE ---
print("Evaluating on Test set...")
test_generator.reset()
preds = model.predict(test_generator, verbose=1)
preds_bin = (preds > 0.5).astype(int)
y_true = test_df[celeb_df.features_name].values

print(f"Accuracy: {np.mean(y_true == preds_bin):.4f}")
print(f"Recall:   {recall_score(y_true, preds_bin, average='macro'):.4f}")
print(f"F1 Score: {f1_score(y_true, preds_bin, average='macro'):.4f}")



Epoch 1/10


I0000 00:00:1767484945.406648     248 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1767484948.683020     245 service.cc:152] XLA service 0x796c86c50650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767484948.683053     245 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1767484948.683057     245 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1767484949.078898     245 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4990/4990 [==============================] - ETA: 0s - loss: 0.3315 - binary_accuracy: 0.8646
Epoch 1: val_binary_accuracy improved from -inf to 0.53513, saving model to /kaggle/working/ckpts_swin/weights-SwinV2-0.5351.h5
4990/4990 [==============================] - 4538s 899ms/step - loss: 0.3315 - binary_accuracy: 0.8646 - val_loss: 2.0315 - val_binary_accuracy: 0.5351
Epoch 2/10
4990/4990 [==============================] - ETA: 0s - loss: 0.2431 - binary_accuracy: 0.8979
Epoch 2: val_binary_accuracy improved from 0.53513 to 0.73531, saving model to /kaggle/working/ckpts_swin/weights-SwinV2-0.7353.h5
4990/4990 [==============================] - 4496s 901ms/step - loss: 0.2431 - binary_accuracy: 0.8979 - val_loss: 0.7759 - val_binary_accuracy: 0.7353
Epoch 3/10
4990/4990 [==============================] - ETA: 0s - loss: 0.2154 - binary_accuracy: 0.9087
Epoch 3: val_binary_accuracy improved from 0.73531 to 0.79285, saving model to /kaggle/working/ckpts_swin/weights-SwinV2-0.7928.h5
49

KeyboardInterrupt: 

In [9]:
ckpt_path = "/kaggle/working/ckpts_swin/weights-SwinV2-0.9239.h5"

model.load_weights(ckpt_path)
print("✅ Loaded checkpoint:", ckpt_path)

✅ Loaded checkpoint: /kaggle/working/ckpts_swin/weights-SwinV2-0.9239.h5


In [10]:
print("Predicting on Test set...")
test_generator.reset()

preds = model.predict(test_generator, verbose=1)
preds_bin = (preds > 0.5).astype(int)

y_true = test_df[celeb_df.features_name].values

print(f"Accuracy: {np.mean(y_true == preds_bin):.4f}")
print(f"Recall:   {recall_score(y_true, preds_bin, average='macro'):.4f}")
print(f"F1 Score: {f1_score(y_true, preds_bin, average='macro'):.4f}")


Predicting on Test set...
94/94 [==============================] - 30s 293ms/step
Accuracy: 0.9240
Recall:   0.5341
F1 Score: 0.5953
